TITLE: M04L1 Notebook

In [ ]:
import pandas as pd
df = pd.read_csv("m4_ecommerce_orders.csv")
df.head()

In [5]:
print(df.shape)
print(df.dtypes)
print(df.isna().sum())
print(df.describe())

(500, 8)
order_id           object
order_date         object
category           object
region             object
quantity            int64
unit_price        float64
shipping_cost     float64
payment_method     object
dtype: object
order_id           0
order_date         0
category           0
region             0
quantity           0
unit_price        21
shipping_cost      0
payment_method     0
dtype: int64
         quantity  unit_price  shipping_cost
count  500.000000  479.000000     500.000000
mean     5.088000  189.523591      12.179560
std      2.653741  115.619862       8.305565
min      1.000000    5.070000     -22.560000
25%      3.000000   87.805000       5.515000
50%      5.000000  177.790000      12.620000
75%      7.000000  293.770000      19.017500
max      9.000000  398.810000      24.980000


In [65]:
# the missing ones, in one line
print(df.isna().sum())

# the impossible ones, one check each. Two are done for you.
print((df['shipping_cost'] < 0).sum(), 'negative shipping costs')
print((df['quantity'] < 1).sum(), 'quantities below one')

# write one more check of your own for another column
print((df['unit_price'].isna()).sum(), 'NA unit costs')
print((df['unit_price'] <= 0).sum(), 'negative or zero unit price')
print(df['order_id'].duplicated().sum(), 'duplicate order ids')

order_id           0
order_date         0
category           0
region             0
quantity           0
unit_price        21
shipping_cost      0
payment_method     0
dtype: int64
13 negative shipping costs
0 quantities below one
21 NA unit costs
0 negative or zero unit price
0 duplicate order ids


In [54]:
negShip = (df['shipping_cost'] < 0)
naPrice = (df['unit_price'].isna())

df[negShip].head()
df[naPrice].head()

,order_id,order_date,category,region,quantity,unit_price,shipping_cost,payment_method
22,ORD00023,2026-09-11,Apparel,South,6,NaN,6.14,gift
45,ORD00046,2026-03-05,Home,South,9,NaN,21.34,paypal
68,ORD00069,2026-04-01,Grocery,Central,1,NaN,8.15,card
91,ORD00092,2026-09-21,Electronics,Central,1,NaN,19.41,gift
114,ORD00115,2026-10-09,Home,East,6,NaN,18.36,card


In [71]:
# run this before you decide which fix to use
before = df['quantity'].mean()
after  = df.dropna(subset=['unit_price'])['quantity'].mean()
print(before, after)
print('change less than 1%', ((before * 0.01) > (before - after)) or ((before * 0.01) < (after - before)))

# then apply your two fixes and save the result as df_clean
df_clean = df.dropna(subset=['unit_price'])
print(df_clean.describe())
df_clean = df_clean[df_clean['shipping_cost'] >= 0]
print(df_clean.describe())
print(df_clean.shape)
print(df_clean.dtypes)
print(df_clean.isna().sum())
print(df[negShip].describe())
print(df[naPrice].describe())
print((df_clean['quantity'] < 1).sum(), 'quantities below one')

5.088 5.079331941544885
change less than 1% True
         quantity  unit_price  shipping_cost
count  479.000000  479.000000     479.000000
mean     5.079332  189.523591      12.209228
std      2.646141  115.619862       8.368581
min      1.000000    5.070000     -22.560000
25%      3.000000   87.805000       5.510000
50%      5.000000  177.790000      12.610000
75%      7.500000  293.770000      19.115000
max      9.000000  398.810000      24.980000
         quantity  unit_price  shipping_cost
count  466.000000  466.000000     466.000000
mean     5.070815  189.180773      12.877039
std      2.643175  115.187110       7.350172
min      1.000000    5.070000       0.000000
25%      3.000000   88.072500       6.032500
50%      5.000000  177.380000      13.060000
75%      7.000000  288.702500      19.180000
max      9.000000  398.810000      24.980000
(466, 8)
order_id           object
order_date         object
category           object
region             object
quantity            int64
un

In [161]:
print(df_clean.groupby('category')['quantity'].mean().sort_values(ascending=False))

category
Toys           5.312500
Electronics    5.213483
Home           5.183673
Apparel        4.966667
Grocery        4.666667
Name: quantity, dtype: float64


In [174]:
print(df_clean.groupby('region')['shipping_cost'].mean().sort_values(ascending=False))

region
South      13.307869
West       13.027647
Central    12.885648
East       12.266667
Name: shipping_cost, dtype: float64


In [159]:
zero_price = df
zero_price['unit_price'] = [df['unit_price'][i] if x is not True else 0 for i,x in enumerate(naPrice)]
#zero_price['unit_price'] = [x if x is not False else 0 for x in zero_price['unit_price']]
zero_price = zero_price[zero_price['shipping_cost'] >= 0]

u_p = [x for x in df_clean['unit_price']]
quan = [x for x in df_clean['quantity']]

df_clean['order_value'] = [x/y for x,y in ((u_p[z],quan[z]) for z in range(len(u_p)))]

u_p = [x for x in zero_price['unit_price']]
quan = [x for x in zero_price['quantity']]

zero_price['order_value'] = [x/y for x,y in ((u_p[z],quan[z]) for z in range(len(u_p)))]

print(zero_price['category'].unique())

averags = [zero_price['order_value'][zero_price['category'] == x].mean() for x in zero_price['category'].unique()]

averags_orig = [df_clean['order_value'][df_clean['category'] == x].mean() for x in df_clean['category'].unique()]

for x,y,z in zip(zero_price['category'].unique(),averags, averags_orig):
    print(x, ' average zeroes = ', y, '\n', x, ' average original = ', z, '\n')

['Home' 'Toys' 'Apparel' 'Electronics' 'Grocery']
Home  average zeroes =  54.7572101210121 
 Home  average original =  56.43345124716552 

Toys  average zeroes =  58.74165585519922 
 Toys  average original =  59.96544035218253 

Apparel  average zeroes =  48.351938880706925 
 Apparel  average original =  52.11264523809524 

Electronics  average zeroes =  49.567896577380964 
 Electronics  average original =  53.466495184590684 

Grocery  average zeroes =  67.45879248120302 
 Grocery  average original =  68.90951920122889 



/var/tmp/braket/ipykernel_46953/1552511408.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  zero_price['order_value'] = [x/y for x,y in ((u_p[z],quan[z]) for z in range(len(u_p)))]


In [171]:
print((zero_price['unit_price'] == 0).sum())

21


In [173]:
print(df_clean.groupby('category')['order_value'].mean().sort_values(ascending=False))
print(zero_price.groupby('category')['order_value'].mean().sort_values(ascending=False))

category
Grocery        68.909519
Toys           59.965440
Home           56.433451
Electronics    53.466495
Apparel        52.112645
Name: order_value, dtype: float64
category
Grocery        67.458792
Toys           58.741656
Home           54.757210
Electronics    49.567897
Apparel        48.351939
Name: order_value, dtype: float64


In [160]:
sorted_averags = zero_price['category'].unique()[sorted(range(len(averags)), key=lambda i: averags[i])]
sorted_averags_orig = zero_price['category'].unique()[sorted(range(len(averags_orig)), key=lambda i: averags_orig[i])]
print(sorted_averags)
print(sorted_averags_orig)

['Apparel' 'Electronics' 'Home' 'Toys' 'Grocery']
['Apparel' 'Electronics' 'Home' 'Toys' 'Grocery']


In [150]:
pd.to_datetime(df_clean['order_date'], errors='raise')

0     2026-04-12
1     2026-03-15
2     2026-10-21
3     2026-03-15
4     2026-03-08
         ...    
495   2026-07-05
496   2026-03-11
497   2026-10-10
498   2026-09-15
499   2026-06-01
Name: order_date, Length: 466, dtype: datetime64[ns]

In [151]:
pd.to_datetime(df['order_date'], errors='raise')

0     2026-04-12
1     2026-03-15
2     2026-10-21
3     2026-03-15
4     2026-03-08
         ...    
495   2026-07-05
496   2026-03-11
497   2026-10-10
498   2026-09-15
499   2026-06-01
Name: order_date, Length: 500, dtype: datetime64[ns]